In [ ]:
# Cell 1: Environment Setup and Library Imports
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, classification_report, roc_auc_score
)

# Plotting configuration
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("All capstone project dependencies imported successfully!")

In [ ]:
# Cell 2: Project 1 - Loan Approval Dataset Generation
np.random.seed(42)
n_samples = 800

# Generating applicant feature vectors
gender = np.random.choice(['Male', 'Female'], size=n_samples, p=[0.8, 0.2])
married = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.65, 0.35])
education = np.random.choice(['Graduate', 'Not Graduate'], size=n_samples, p=[0.75, 0.25])
self_employed = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.15, 0.85])

applicant_income = np.random.randint(1500, 25000, size=n_samples)
coapplicant_income = np.random.choice([0, np.random.randint(1000, 10000)], size=n_samples, p=[0.3, 0.7])
loan_amount = np.random.randint(50, 600, size=n_samples) # in thousands
loan_amount_term = np.random.choice([120, 180, 240, 360], size=n_samples, p=[0.05, 0.1, 0.1, 0.75])
credit_history = np.random.choice([1.0, 0.0], size=n_samples, p=[0.82, 0.18])
property_area = np.random.choice(['Urban', 'Semiurban', 'Rural'], size=n_samples, p=[0.33, 0.34, 0.33])

# Synthetic target generation rule (Credit History & Income heavy weighting)
score = (credit_history * 4.5) + (applicant_income / 5000) + (coapplicant_income / 4000) - (loan_amount / 150)
loan_status = np.where(score > 2.5, 'Y', 'N')

df_loan = pd.DataFrame({
    'Loan_ID': [f'LP{i+1000:04d}' for i in range(n_samples)],
    'Gender': gender,
    'Married': married,
    'Education': education,
    'Self_Employed': self_employed,
    'ApplicantIncome': applicant_income,
    'CoapplicantIncome': coapplicant_income,
    'LoanAmount': loan_amount,
    'Loan_Amount_Term': loan_amount_term,
    'Credit_History': credit_history,
    'Property_Area': property_area,
    'Loan_Status': loan_status
})

df_loan.to_csv("loan_approval_dataset.csv", index=False)
print("Project 1 Loan Dataset Shape:", df_loan.shape)
df_loan.head()

In [ ]:
# Cell 3: Project 1 - Preprocessing Pipeline and Model Comparison
X_loan = df_loan.drop(columns=['Loan_ID', 'Loan_Status'])
y_loan = df_loan['Loan_Status'].map({'Y': 1, 'N': 0})

# Train-Test Split
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_loan, y_loan, test_size=0.2, random_state=42, stratify=y_loan
)

# Preprocessing Transformers
num_cols_loan = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
cat_cols_loan = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']

preprocessor_loan = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_loan),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols_loan)
    ]
)

# Classification Models
classifiers_loan = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results_loan = {}

for name, clf in classifiers_loan.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor_loan),
        ('classifier', clf)
    ])
    
    pipe.fit(X_train_l, y_train_l)
    y_pred = pipe.predict(X_test_l)
    y_proba = pipe.predict_proba(X_test_l)[:, 1]
    
    results_loan[name] = {
        'Accuracy': accuracy_score(y_test_l, y_pred),
        'Precision': precision_score(y_test_l, y_pred),
        'Recall': recall_score(y_test_l, y_pred),
        'F1-Score': f1_score(y_test_l, y_pred),
        'ROC-AUC': roc_auc_score(y_test_l, y_proba),
        'Pipeline': pipe
    }

df_res_loan = pd.DataFrame(results_loan).T[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]
print("--- Project 1 Algorithm Comparison ---")
print(df_res_loan)

# Save Best Model Pipeline
best_loan_model_name = df_res_loan['F1-Score'].idxmax()
best_loan_pipeline = results_loan[best_loan_model_name]['Pipeline']
joblib.dump(best_loan_pipeline, "loan_approval_pipeline.pkl")
print(f"\nSaved best model ({best_loan_model_name}) to 'loan_approval_pipeline.pkl'")

In [ ]:
# Cell 4: Project 2 - Disease Prediction Dataset Generation
np.random.seed(101)
n_patients = 700

diseases = ['Diabetes', 'Hypertension', 'Heart Disease', 'Healthy']

fever = np.random.choice([0, 1], size=n_patients, p=[0.7, 0.3])
cough = np.random.choice([0, 1], size=n_patients, p=[0.6, 0.4])
fatigue = np.random.choice([0, 1], size=n_patients, p=[0.5, 0.5])
chest_pain = np.random.choice([0, 1], size=n_patients, p=[0.8, 0.2])
blood_sugar = np.random.uniform(70, 250, size=n_patients)
bmi = np.random.uniform(18.5, 40.0, size=n_patients)
sys_bp = np.random.uniform(90, 180, size=n_patients)
age = np.random.randint(20, 80, size=n_patients)

# Target generation rules
target = []
for i in range(n_patients):
    if blood_sugar[i] > 140 or bmi[i] > 30.0:
        target.append('Diabetes')
    elif sys_bp[i] > 135:
        target.append('Hypertension')
    elif chest_pain[i] == 1 and age[i] > 45:
        target.append('Heart Disease')
    else:
        target.append('Healthy')

df_disease = pd.DataFrame({
    'Patient_ID': [f'PAT_{i+5000:04d}' for i in range(n_patients)],
    'Age': age,
    'Fever': fever,
    'Cough': cough,
    'Fatigue': fatigue,
    'Chest_Pain': chest_pain,
    'Fasting_Blood_Sugar': np.round(blood_sugar, 1),
    'BMI': np.round(bmi, 1),
    'Systolic_BP': np.round(sys_bp, 1),
    'Diagnosis': target
})

df_disease.to_csv("disease_prediction_dataset.csv", index=False)
print("Project 2 Disease Dataset Shape:", df_disease.shape)
df_disease.head()

In [ ]:
# Cell 5: Project 2 - Disease Classification Pipeline & Evaluation
X_disease = df_disease.drop(columns=['Patient_ID', 'Diagnosis'])
y_disease = df_disease['Diagnosis']

# Train-Test Split
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_disease, y_disease, test_size=0.2, random_state=42, stratify=y_disease
)

# Feature Scaling
scaler_disease = StandardScaler()
X_train_d_scaled = scaler_disease.fit_transform(X_train_d)
X_test_d_scaled = scaler_disease.transform(X_test_d)

# Classifiers
disease_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42)
}

res_disease = {}

for name, model in disease_models.items():
    model.fit(X_train_d_scaled, y_train_d)
    y_pred = model.predict(X_test_d_scaled)
    
    res_disease[name] = {
        'Accuracy': accuracy_score(y_test_d, y_pred),
        'Macro F1': f1_score(y_test_d, y_pred, average='macro'),
        'Model': model
    }

df_res_disease = pd.DataFrame(res_disease).T[['Accuracy', 'Macro F1']]
print("--- Project 2 Disease Model Comparison ---")
print(df_res_disease)

# Plot Confusion Matrix for Best Model
best_disease_name = df_res_disease['Macro F1'].idxmax()
best_disease_model = res_disease[best_disease_name]['Model']

y_pred_best_d = best_disease_model.predict(X_test_d_scaled)
cm = confusion_matrix(y_test_d, y_pred_best_d)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=np.unique(y_disease), yticklabels=np.unique(y_disease))
plt.title(f'Disease Diagnosis Confusion Matrix ({best_disease_name})')
plt.xlabel('Predicted Diagnosis')
plt.ylabel('Actual Diagnosis')
plt.tight_layout()
plt.show()

# Save Scaler & Disease Prediction Model
joblib.dump(best_disease_model, "disease_prediction_model.pkl")
joblib.dump(scaler_disease, "disease_scaler.pkl")
print(f"Saved best model ({best_disease_name}) to 'disease_prediction_model.pkl'")

In [ ]:
# Cell 6: Generate Streamlit Prediction Web Application Code
app_code = """import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.set_page_config(page_title="MCA Capstone ML Suite", layout="wide")

st.title("🎓 MCA Capstone ML Application Dashboard")
st.write("Session 2026–2027 | Lab Sheet-10 Capstone Suite")

# Navigation Sidebar
option = st.sidebar.selectbox(
    "Choose Capstone Application",
    ["1. Loan Approval Prediction System", "2. Disease Prediction System"]
)

if option == "1. Loan Approval Prediction System":
    st.header("💳 Loan Approval Prediction System")
    st.write("Enter applicant details below to test the trained machine learning pipeline.")
    
    try:
        loan_pipeline = joblib.load("loan_approval_pipeline.pkl")
        
        col1, col2 = st.columns(2)
        with col1:
            gender = st.selectbox("Gender", ["Male", "Female"])
            married = st.selectbox("Married", ["Yes", "No"])
            education = st.selectbox("Education", ["Graduate", "Not Graduate"])
            self_employed = st.selectbox("Self Employed", ["Yes", "No"])
            property_area = st.selectbox("Property Area", ["Urban", "Semiurban", "Rural"])
            
        with col2:
            app_income = st.number_input("Applicant Income ($)", min_value=1000, max_value=50000, value=5000)
            coapp_income = st.number_input("Co-applicant Income ($)", min_value=0, max_value=25000, value=1500)
            loan_amt = st.number_input("Loan Amount (in Thousands $)", min_value=10, max_value=1000, value=150)
            loan_term = st.selectbox("Loan Term (Months)", [120, 180, 240, 360])
            credit_hist = st.selectbox("Credit History Clear?", [1.0, 0.0])

        if st.button("Predict Loan Status"):
            input_df = pd.DataFrame([{
                'Gender': gender,
                'Married': married,
                'Education': education,
                'Self_Employed': self_employed,
                'ApplicantIncome': app_income,
                'CoapplicantIncome': coapp_income,
                'LoanAmount': loan_amt,
                'Loan_Amount_Term': loan_term,
                'Credit_History': credit_hist,
                'Property_Area': property_area
            }])
            
            prediction = loan_pipeline.predict(input_df)[0]
            probability = loan_pipeline.predict_proba(input_df)[0][1]
            
            if prediction == 1:
                st.success(f"✅ **Loan Approved!** (Approval Probability: {probability*100:.2f}%)")
            else:
                st.error(f"❌ **Loan Rejected.** (Approval Probability: {probability*100:.2f}%)")
                
    except FileNotFoundError:
        st.warning("Please run the training notebook cells first to generate 'loan_approval_pipeline.pkl'.")

elif option == "2. Disease Prediction System":
    st.header("🩺 Disease Prediction System")
    st.write("Input patient clinical indicators and symptoms to obtain automated medical diagnostic screening.")
    
    try:
        disease_model = joblib.load("disease_prediction_model.pkl")
        disease_scaler = joblib.load("disease_scaler.pkl")
        
        col1, col2 = st.columns(2)
        with col1:
            age = st.slider("Patient Age", 1, 100, 45)
            fever = st.selectbox("Fever Present?", [0, 1], format_func=lambda x: "Yes" if x==1 else "No")
            cough = st.selectbox("Cough Present?", [0, 1], format_func=lambda x: "Yes" if x==1 else "No")
            fatigue = st.selectbox("Fatigue Present?", [0, 1], format_func=lambda x: "Yes" if x==1 else "No")
            
        with col2:
            chest_pain = st.selectbox("Chest Pain Present?", [0, 1], format_func=lambda x: "Yes" if x==1 else "No")
            blood_sugar = st.number_input("Fasting Blood Sugar (mg/dL)", 60.0, 300.0, 110.0)
            bmi = st.number_input("Body Mass Index (BMI)", 15.0, 50.0, 24.5)
            sys_bp = st.number_input("Systolic Blood Pressure (mmHg)", 80.0, 200.0, 120.0)

        if st.button("Predict Clinical Diagnosis"):
            raw_input = np.array([[age, fever, cough, fatigue, chest_pain, blood_sugar, bmi, sys_bp]])
            scaled_input = disease_scaler.transform(raw_input)
            
            pred_disease = disease_model.predict(scaled_input)[0]
            
            if pred_disease == "Healthy":
                st.success(f"🎉 **Diagnosis:** {pred_disease} (No major risk indicators detected)")
            else:
                st.warning(f"⚠️ **Predicted Diagnosis:** {pred_disease}")
                
    except FileNotFoundError:
        st.warning("Please run the training notebook cells first to generate 'disease_prediction_model.pkl' and 'disease_scaler.pkl'.")
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Streamlit UI Application file 'app.py' successfully generated!")